[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session5/solutions/Lab2_Tools_Agent_Loop_Solutions.ipynb)

# Lab 2: Tools & the Agent Loop — SOLUTIONS
## CCI Session 5

**Duration:** 15 minutes

### Clinical Scenario
> Build a clinical assistant with tools that look up patient data, check vitals reference ranges, and flag drug interactions. Compare with the manual tool calling you built in Session 3 Lab 3.

### Objective
- Define clinical tools with `@tool` decorator
- Build an agent with multiple tools
- Observe the agent loop in action
- Add error handling and guardrails

---
## Setup

In [ ]:
!pip install -q langchain langchain-openai langgraph

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup complete.")

---
## Cell 1 — Define Clinical Tools with @tool

In Session 3, you defined tools as JSON schemas and wrote manual dispatch logic:
```python
tools = [{
    "type": "function",
    "function": {
        "name": "lookup_patient",
        "parameters": { ... }  # Manual JSON schema
    }
}]
```

With LangChain's `@tool` decorator, the schema is generated automatically from the function signature and docstring.

In [ ]:
from langchain_core.tools import tool

# --- Tool 1: Patient Lookup ---
@tool
def lookup_patient(mrn: str) -> dict:
    """Look up a patient record by their Medical Record Number (MRN).
    Returns patient demographics and current diagnosis."""
    patients = {
        "MRN001": {
            "name": "Ahmad Al-Rashid",
            "age": 58,
            "diagnosis": "Non-Small Cell Lung Cancer (NSCLC), Stage IIIA",
            "current_medications": ["cisplatin", "pemetrexed", "ondansetron"],
            "allergies": ["penicillin"]
        },
        "MRN002": {
            "name": "Fatima Hassan",
            "age": 45,
            "diagnosis": "Chronic Myeloid Leukemia (CML), Chronic Phase",
            "current_medications": ["imatinib", "allopurinol"],
            "allergies": []
        },
        "MRN003": {
            "name": "Yousef Bakri",
            "age": 67,
            "diagnosis": "Diffuse Large B-Cell Lymphoma (DLBCL), Stage II",
            "current_medications": ["rituximab", "cyclophosphamide", "doxorubicin", "vincristine", "prednisone"],
            "allergies": ["sulfa"]
        }
    }
    patient = patients.get(mrn.upper())
    if patient:
        return patient
    return {"error": f"Patient with MRN {mrn} not found"}

# --- Tool 2: Vitals Check --- SOLUTION
@tool
def check_vitals(mrn: str) -> dict:
    """Check the latest vital signs for a patient by MRN.
    Returns temperature, heart rate, blood pressure, WBC, and platelet count."""
    vitals = {
        "MRN001": {
            "temperature_c": 38.5,
            "heart_rate_bpm": 92,
            "blood_pressure": "130/85",
            "wbc_k_per_ul": 3.2,
            "platelets_k_per_ul": 145,
            "flags": ["Fever (>38.0C)", "Low WBC (neutropenia risk)"]
        },
        "MRN002": {
            "temperature_c": 36.8,
            "heart_rate_bpm": 78,
            "blood_pressure": "120/80",
            "wbc_k_per_ul": 8.5,
            "platelets_k_per_ul": 210,
            "flags": []
        },
        "MRN003": {
            "temperature_c": 37.1,
            "heart_rate_bpm": 85,
            "blood_pressure": "140/90",
            "wbc_k_per_ul": 2.8,
            "platelets_k_per_ul": 95,
            "flags": ["Low WBC (neutropenia risk)", "Low platelets (thrombocytopenia)", "Elevated BP"]
        }
    }
    patient_vitals = vitals.get(mrn.upper())
    if patient_vitals:
        return patient_vitals
    return {"error": f"No vitals found for MRN {mrn}"}

# --- Tool 3: Drug Interaction Check --- SOLUTION
@tool
def check_drug_interaction(drug1: str, drug2: str) -> dict:
    """Check for known drug-drug interactions between two medications.
    Returns severity level and clinical description."""
    interactions = {
        ("cisplatin", "pemetrexed"): {
            "severity": "moderate",
            "description": "Monitor renal function closely. Cisplatin nephrotoxicity may increase pemetrexed levels."
        },
        ("imatinib", "allopurinol"): {
            "severity": "low",
            "description": "No clinically significant interaction expected."
        },
        ("rituximab", "cyclophosphamide"): {
            "severity": "moderate",
            "description": "Increased immunosuppression risk. Monitor for infections."
        },
        ("methotrexate", "ibuprofen"): {
            "severity": "high",
            "description": "NSAID reduces methotrexate clearance. Risk of severe toxicity."
        },
        ("doxorubicin", "cyclophosphamide"): {
            "severity": "moderate",
            "description": "Increased cardiotoxicity risk. Monitor cardiac function."
        }
    }
    # Check both orderings
    key1 = (drug1.lower(), drug2.lower())
    key2 = (drug2.lower(), drug1.lower())
    result = interactions.get(key1) or interactions.get(key2)
    if result:
        return {"drug1": drug1, "drug2": drug2, **result}
    return {"drug1": drug1, "drug2": drug2, "severity": "none", "description": "No known interaction in database."}

# Verify tools are properly defined
tools = [lookup_patient, check_vitals, check_drug_interaction]
for t in tools:
    print(f"Tool: {t.name}")
    print(f"  Description: {t.description}")
    print(f"  Schema: {t.args_schema.model_json_schema()}")
    print()

---
## Cell 2 — Create the Agent

Now let's create an agent that can use all three tools. We'll use `create_react_agent` which implements the ReAct (Reason + Act) pattern.

In [ ]:
from langgraph.prebuilt import create_react_agent

# System prompt to set clinical context
system_prompt = """You are a clinical decision support assistant at King Hussein Cancer Center (KHCC).
You have access to patient records, vitals, and drug interaction databases.
Always verify patient identity before providing clinical information.
Flag any concerning vitals or dangerous drug interactions prominently."""

# SOLUTION: Create the agent
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

print("Agent created with", len(tools), "tools")

---
## Cell 3 — Invoke and Observe the Agent Loop

Watch how the agent reasons about which tools to call and in what order.

In [ ]:
# Ask a question that requires multiple tool calls
query = "Look up patient MRN001 and check their vitals. Are any vitals concerning?"

# SOLUTION: Invoke the agent
result = agent.invoke({"messages": [{"role": "user", "content": query}]})

# Print the full agent loop
print("=" * 60)
print("AGENT LOOP TRACE")
print("=" * 60)

for i, msg in enumerate(result["messages"]):
    role = msg.__class__.__name__
    print(f"\n--- Step {i}: [{role}] ---")

    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  TOOL CALL: {tc['name']}({tc['args']})")

    if msg.content:
        content = msg.content if len(msg.content) < 500 else msg.content[:500] + "..."
        print(f"  {content}")

---
## Cell 4 — Multi-Turn Conversation

The agent maintains context across turns. Let's ask follow-up questions.

In [ ]:
# SOLUTION: Build on previous conversation with a follow-up
follow_up_messages = result["messages"] + [
    {"role": "user", "content": "Check if there are any interactions between their current medications cisplatin and pemetrexed."}
]

# SOLUTION: Invoke the agent with the follow-up
result2 = agent.invoke({"messages": follow_up_messages})

# Print only the new messages (after the previous conversation)
new_messages = result2["messages"][len(result["messages"]):]
print("=" * 60)
print("FOLLOW-UP RESPONSE")
print("=" * 60)

for msg in new_messages:
    role = msg.__class__.__name__
    print(f"\n[{role}]")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  TOOL CALL: {tc['name']}({tc['args']})")
    if msg.content:
        print(f"  {msg.content[:500]}")

---
## Cell 5 — Iteration Limits and Error Handling

In production, you want to prevent infinite loops. Let's add safety guardrails.

In [ ]:
from langgraph.errors import GraphRecursionError

# SOLUTION: Invoke with recursion_limit
try:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Look up MRN001, check vitals, and check all drug interactions between their medications."}]},
        config={"recursion_limit": 10}
    )
    # Print final answer
    final_msg = result["messages"][-1]
    print("Final answer:")
    print(final_msg.content[:500])
    print(f"\nTotal messages in loop: {len(result['messages'])}")

except GraphRecursionError:
    print("Agent hit the recursion limit! The task may be too complex.")
    print("Consider breaking it into smaller queries.")

---
## Cell 6 — Side-by-Side Comparison: Session 3 vs LangChain

Let's see how much simpler the LangChain approach is compared to Session 3.

In [ ]:
# This cell is for READING — don't run it (it uses the raw OpenAI client)
# It shows the Session 3 approach side-by-side with the LangChain approach

comparison = """
╔══════════════════════════════════════╦══════════════════════════════════════╗
║  SESSION 3: Raw OpenAI API          ║  SESSION 5: LangChain + LangGraph   ║
╠══════════════════════════════════════╬══════════════════════════════════════╣
║                                      ║                                      ║
║  # 1. Define tool schema manually    ║  # 1. Just use @tool decorator       ║
║  tools = [{                          ║  @tool                               ║
║    "type": "function",               ║  def lookup_patient(mrn: str) -> d:  ║
║    "function": {                     ║      \"\"\"Docstring = schema.\"\"\"   ║
║      "name": "lookup_patient",       ║      return patients[mrn]            ║
║      "parameters": {                 ║                                      ║
║        "type": "object",             ║                                      ║
║        "properties": { ... }         ║                                      ║
║      }                               ║                                      ║
║    }                                 ║                                      ║
║  }]                                  ║                                      ║
║                                      ║                                      ║
║  # 2. Manual tool dispatch           ║  # 2. Agent handles dispatch         ║
║  while True:                         ║  agent = create_react_agent(         ║
║    resp = client.chat.completions.   ║      model=llm,                      ║
║      create(...)                     ║      tools=[lookup_patient]          ║
║    if resp...tool_calls:             ║  )                                   ║
║      for tc in tool_calls:           ║                                      ║
║        name = tc.function.name       ║  # 3. Just invoke                    ║
║        args = json.loads(...)        ║  result = agent.invoke(              ║
║        if name == "lookup":          ║      {"messages": [query]}           ║
║          result = lookup(...)        ║  )                                   ║
║        messages.append(...)          ║                                      ║
║    else:                             ║                                      ║
║      break                           ║                                      ║
║                                      ║                                      ║
║  ~40 lines of code                   ║  ~10 lines of code                   ║
╚══════════════════════════════════════╩══════════════════════════════════════╝
"""
print(comparison)

---
## Stretch Challenge

1. Add a 4th tool: `get_lab_results(mrn: str) -> dict` with mock CBC/chemistry data
2. Ask the agent: "Give me a complete clinical summary for MRN003 including labs, vitals, and any drug interactions"
3. Count how many tool calls the agent makes

### KHCC Connection
This pattern directly maps to a real clinical decision support system at KHCC:
- `lookup_patient` could connect to the Hospital Information System (HIS)
- `check_vitals` could pull from bedside monitors or nursing documentation
- `check_drug_interaction` could connect to a drug database like Lexicomp
- The agent loop automates the workflow a pharmacist does manually today

In [ ]:
# STRETCH SOLUTION

@tool
def get_lab_results(mrn: str) -> dict:
    """Retrieve the latest laboratory results for a patient by MRN.
    Returns CBC and basic metabolic panel results."""
    labs = {
        "MRN001": {
            "hemoglobin": 11.2,
            "hematocrit": 33.5,
            "creatinine": 1.4,
            "bun": 28,
            "sodium": 138,
            "potassium": 4.1,
            "flags": ["Low hemoglobin", "Elevated creatinine"]
        },
        "MRN002": {
            "hemoglobin": 13.5,
            "hematocrit": 40.2,
            "creatinine": 0.9,
            "bun": 15,
            "sodium": 140,
            "potassium": 4.3,
            "flags": []
        },
        "MRN003": {
            "hemoglobin": 9.8,
            "hematocrit": 29.4,
            "creatinine": 1.1,
            "bun": 22,
            "sodium": 136,
            "potassium": 3.8,
            "flags": ["Low hemoglobin (anemia)", "Low hematocrit"]
        }
    }
    patient_labs = labs.get(mrn.upper())
    if patient_labs:
        return patient_labs
    return {"error": f"No lab results found for MRN {mrn}"}

# Rebuild agent with all 4 tools
all_tools = [lookup_patient, check_vitals, check_drug_interaction, get_lab_results]

agent_v2 = create_react_agent(
    model=llm,
    tools=all_tools,
    prompt=system_prompt
)

result = agent_v2.invoke(
    {"messages": [{"role": "user", "content": "Give me a complete clinical summary for MRN003 including labs, vitals, and any drug interactions."}]},
    config={"recursion_limit": 20}
)

# Count tool calls
tool_call_count = 0
for msg in result["messages"]:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        tool_call_count += len(msg.tool_calls)

print(f"Total tool calls: {tool_call_count}")
print(f"Total messages: {len(result['messages'])}")
print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(result["messages"][-1].content)